In [1]:
import torchvision
import torch
from jetbot import Robot
from IPython.display import display
import ipywidgets
import traitlets
from jetbot import Camera, bgr8_to_jpeg
import time

# our imports
from collisionAvoidance import CollisionAvoidance
from roadFollowing import RoadFollowing

In [2]:
roadFollowingModel = torchvision.models.resnet18(pretrained=False)
roadFollowingModel.fc = torch.nn.Linear(512, 2)

# adjust this model to use the road following training data
roadFollowingModel.load_state_dict(torch.load('best_steering_model_xy.pth'))

<All keys matched successfully>

In [ ]:
collisionAvoidanceModel = torchvision.models.resnet18(pretrained=False)
collisionAvoidanceModel.fc = torch.nn.Linear(512, 2)

# adjust this model to use the collision avoidance training data
collisionAvoidanceModel.load_state_dict(torch.load('best_model_resnet18.pth'))

In [3]:
# create the camera object and display it as a widget
camera = Camera()
image_widget = ipywidgets.Image()
traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)
display(image_widget)

# create a robot instance
robot = Robot()

rf = RoadFollowing(roadFollowingModel, robot)
ca = CollisionAvoidance(collisionAvoidanceModel, robot)

#def startRobot(change):
#    rf.execute(change)
#    camera.observe(rf.execute, names='value')
#    
#    ca.update(change)
#    camera.observe(ca.update, names='value')
    
def robotLoop(change):
    if ca.update(change):
        robot.stop()
    else:
        motorValues = rf.execute(change)
        robot.left_motor.value = motorValues[0]
        robot.right_motor.value = motorValues[1]
        

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [4]:
camera.observe(robotLoop, names='value')

In [5]:
def stopRobot():
    camera.unobserve(robotLoop, names='value')

    # sleep to make sure frames process before killing everything
    time.sleep(0.1)
    robot.stop()
    
    camera.stop()
    
stopRobot()